In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
import dagshub
dagshub.init(repo_owner='AMR-ITH', repo_name='RealEstateInsights', mlflow=True)
import mlflow

# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow")

# mlflow experiment

mlflow.set_experiment("Exp 2 - model selection-30")


Accessing as AMR-ITH

Initialized MLflow to track repo "AMR-ITH/RealEstateInsights"

Repository AMR-ITH/RealEstateInsights initialized!

<Experiment: artifact_location='mlflow-artifacts:/9d8b0d6af2e14e9db012af318ad26754', creation_time=1746518661149, experiment_id='3', last_update_time=1746518661149, lifecycle_stage='active', name='Exp 2 - model selection-30', tags={}>

In [4]:
# pathlib is a module in Python that provides an object-oriented interface 
# for working with file system paths.
current = Path.cwd()
parent = current.parent

# load the dat 
df = pd.read_csv(parent /'data-scraped/interim_data.csv')
df.head()



,apartment_name,appartment_loc,zone,bhk_type,construction_status,carpet_area,bulit_area,super_bulit_area,price_value,nearbylocation,facility,luxury_facility_scores
0,nambiar millennia,Sarjapur Road,east,1,Under Construction,NaN,668.0,NaN,0.53,"[('mahatma vidhyalaya', '400 m'), ('eterssrt m...","['Yoga/Meditation Area', ""Children's Play Area...",69
1,provident capella,Samethanahalli,east,1,New Property,NaN,431.0,480.0,0.55,"[('soukya road', '1.4 km'), ('mvj college of e...","[""Children's Play Area"", 'Creche/Day Care', 'J...",70
2,brigade citre budigere cross,Byrathi,east,1,New Property,NaN,619.0,689.0,0.69,"[('one world international school', '3.2kms'),...","['Pet Park', ""Children's Play Area"", 'Landscap...",50
3,sattva east crest bandapura,Budigere Cross,east,1,New Property,NaN,537.0,598.0,0.70,"[('prerana international school', '700 m'), ('...","['Banquet Hall', 'Creche/Day Care', ""Children'...",74
4,sowparnika columns,Soukya Road,east,1,New Property,486.0,619.0,736.0,0.52,"['Whitefield Kadugodi Metro Station', 'Nexus S...","['Lift(s)', 'Swimming Pool', 'Park', 'Fitness ...",44


In [5]:
def categorize_luxury(score):
    if 0 <= score < 50:
        return 'low'
    elif 50 <= score < 150:
        return 'medium'
    else:
        return 'high'
    
df['luxury_category'] = df['luxury_facility_scores'].apply(categorize_luxury)

In [6]:
df.drop(columns=['carpet_area','super_bulit_area','nearbylocation','facility','apartment_name','appartment_loc','luxury_facility_scores'], inplace=True)

In [7]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value'])
y = temp_df['price_value']

In [8]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [14]:
# do the basic processing input data

num_cols = ['bulit_area']
nomial_cols = ['zone']
ordinal_cols = ['construction_status','bhk_type','luxury_category']


In [15]:
bhk_type_order = ['1','2','3','4','5','6','7','8','9','10']
construction_status_order = ['New Property','Under Construction', 'Relatively New', 'Moderatly Old', 'Old','undefined']
luxury_facility_scores_order = ['low','medium','high']

In [16]:
# build a preprocessor

prepocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nomial_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[construction_status_order,bhk_type_order,luxury_facility_scores_order]), ordinal_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

prepocessor.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(), ['bulit_area']),
                                ('nominal_encode',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['zone']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['New Property',
                                                             'Under '
                                                             'Construction',
                                                             'Relatively New',
                                                             'Moderatly Old',
                                                             'Old',
                                                             'undefined'],
                                                            ['1', '2', '3', '4',
                                                             '5', '6', '7', '8',
                                                             '9', '10'],
                                                            ['low', 'medium',
                                                             'high']]),
                                 ['construction_status', 'bhk_type',
                                  'luxury_category'])],
                  verbose_feature_names_out=False)

In [17]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type,luxury_category
2842,0.197885,0.0,1.0,0.0,0.0,0.0,2.0,1.0
903,0.185153,1.0,0.0,0.0,0.0,2.0,2.0,1.0
3262,0.297475,0.0,1.0,0.0,0.0,2.0,2.0,1.0
109,0.114804,1.0,0.0,0.0,0.0,1.0,1.0,1.0
5602,0.092361,0.0,0.0,0.0,1.0,5.0,1.0,0.0
...,...,...,...,...,...,...,...,...
3772,0.224860,0.0,1.0,0.0,0.0,0.0,3.0,1.0
5191,0.140160,0.0,0.0,1.0,0.0,0.0,2.0,1.0
5226,0.175874,0.0,0.0,1.0,0.0,4.0,2.0,1.0
5390,0.118041,0.0,0.0,0.0,1.0,2.0,1.0,1.0


In [18]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error, r2_score


c:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical("model",["SVM","RF","KNN","GB","XGB","LGBM"])

        if model_name == "SVM":
            kernel_svm = trial.suggest_categorical("kernel_svm",["linear","poly","rbf"])
            if kernel_svm == "linear":
                c_linear = trial.suggest_float("c_linear",0.1,10)  
                model = SVR(C=c_linear,kernel="linear")

            elif kernel_svm == "poly":
                c_poly = trial.suggest_float("c_poly",0.1,10)  
                degree_poly = trial.suggest_int("degree_poly",1,5)
                model = SVR(C=c_poly,degree=degree_poly,
                            kernel="poly")

            else:
                c_rbf = trial.suggest_float("c_rbf",0.1,100)  
                gamma_rbf = trial.suggest_float("gamma_rbf",0.001,10)  
                model = SVR(C=c_rbf,gamma=gamma_rbf,
                            kernel="rbf")

        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,200)
            max_depth_rf = trial.suggest_int("max_depth_rf",2,20)
            model = RandomForestRegressor(n_estimators=n_estimators_rf,
                                        max_depth=max_depth_rf,
                                        random_state=42,
                                        n_jobs=-1)

        elif model_name == "GB":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,200)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",0.01,1)  
            max_depth_gb = trial.suggest_int("max_depth_gb",2,20)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb,
                                                learning_rate=learning_rate_gb,
                                                max_depth=max_depth_gb,
                                                random_state=42)

        elif model_name == "KNN":
            n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,25)
            weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
            model = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                        weights=weights_knn,n_jobs=-1)

        elif model_name == "XGB":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,200)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",0.01,0.5)  
            max_depth_xgb = trial.suggest_int("max_depth_xgb",2,20)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                    learning_rate=learning_rate_xgb,
                                    max_depth=max_depth_xgb,
                                    random_state=42,
                                    n_jobs=-1)

        elif model_name == "LGBM":
            n_estimators_lgbm = trial.suggest_int("n_estimators_lgbm",10,200)
            learning_rate_lgbm = trial.suggest_float("learning_rate_lgbm",0.01,0.5)  
            max_depth_lgbm = trial.suggest_int("max_depth_lgbm",2,20)
            model = LGBMRegressor(n_estimators=n_estimators_lgbm,
                                    learning_rate=learning_rate_lgbm,
                                    max_depth=max_depth_lgbm,
                                    random_state=42)
            
        
        # train the model
        model.fit(X_train_trans,y_train)

        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)

        # calculate the test mae,r2
        mae = mean_absolute_error(y_test,y_pred_test)
        r2 = r2_score(y_test,y_pred_test)
        
        # Handle None or NaN values (fix for float casting error)
        if mae is None or np.isnan(mae):
            mae = float('inf')
        if r2 is None or np.isnan(r2):
            r2 = -float('inf')
        
        # log the metrics
        mlflow.log_metric("mae-test",mae)
        mlflow.log_metric("r2-test",r2)
        
        # calculate the train mae,r2
        mae_train = mean_absolute_error(y_train,y_pred_train)
        r2_train = r2_score(y_train,y_pred_train)
        
        # Handle None or NaN values (fix for float casting error)
        if mae_train is None or np.isnan(mae_train):
            mae_train = float('inf')
        if r2_train is None or np.isnan(r2_train):
            r2_train = -float('inf')

        mlflow.log_metric("mae-train",mae_train)
        mlflow.log_metric("r2-train",r2_train)

        # Add model signature and example (fix for the warnings)
        signature = mlflow.models.infer_signature(X_train_trans, y_pred_train)
        
        # log the model
        mlflow.sklearn.log_model(model, "model", signature=signature)
        
        # log the params
        mlflow.log_params(model.get_params())
        
        return mae  # Return the metric being optimized

In [20]:
import time


# Configure MLflow retry settings to avoid 429 errors
mlflow.tracking.DEFAULT_TRACKING_URI = "https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow"
# Set a delay between requests to avoid rate limiting
REQUEST_DELAY = 1  # Add a 1-second delay between requests

# Import the objective function from above

# Create study with fewer trials and add delay 
study = optuna.create_study(direction="minimize", study_name="model_selection")

# Main optimization with rate limiting
with mlflow.start_run(run_name="Best Model") as parent:
    # Run fewer trials and add delay between them to avoid rate limiting
    for _ in range(30):  
        try:
            study.optimize(objective, n_trials=1)  # Run just one trial at a time
            time.sleep(REQUEST_DELAY)  # Add delay between trials
        except Exception as e:
            if "429" in str(e):  # If hit by rate limit
                print(f"Rate limit hit. Waiting for 5 seconds before retry.")
                time.sleep(5)  # Wait longer if rate limited
            else:
                print(f"Error: {e}")
                # Continue running other trials even if one fails
                
    # log the best parameters
    if study.best_params:
        mlflow.log_params(study.best_params)
        # log the best score
        mlflow.log_metric("best_score", study.best_value)

[I 2025-05-12 08:56:14,513] A new study created in memory with name: model_selection


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000368 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 276
[LightGBM] [Info] Number of data points in the train set: 4860, number of used features: 8
[LightGBM] [Info] Start training from score 1.964580
🏃 View run zealous-steed-11 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3/runs/3f084844ab5d4912b6ce55bbc0127fe7
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3


[I 2025-05-12 08:56:43,524] Trial 0 finished with value: 0.48014320357578505 and parameters: {'model': 'LGBM', 'n_estimators_lgbm': 140, 'learning_rate_lgbm': 0.09925065998478537, 'max_depth_lgbm': 19}. Best is trial 0 with value: 0.48014320357578505.


🏃 View run overjoyed-steed-230 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3/runs/6a78f5265bb847c39a0fe8ac00354768
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3


[I 2025-05-12 08:57:00,400] Trial 1 finished with value: 0.5361398873427278 and parameters: {'model': 'XGB', 'n_estimators_xgb': 152, 'learning_rate_xgb': 0.4313794049367109, 'max_depth_xgb': 12}. Best is trial 0 with value: 0.48014320357578505.


...

🏃 View run adventurous-frog-837 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3/runs/e83e3298b71d40ac8660f5194b4be7d7
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3


[I 2025-05-12 09:08:43,070] Trial 28 finished with value: 0.45945120985055626 and parameters: {'model': 'RF', 'n_estimators_rf': 133, 'max_depth_rf': 13}. Best is trial 24 with value: 0.45875272767399033.


🏃 View run redolent-crane-957 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3/runs/83a6b6bca4bb459e93df521212532531
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3


[I 2025-05-12 09:09:09,761] Trial 29 finished with value: 0.45991933344771063 and parameters: {'model': 'RF', 'n_estimators_rf': 142, 'max_depth_rf': 13}. Best is trial 24 with value: 0.45875272767399033.


🏃 View run Best Model at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3/runs/e4dd0c081d3a4443ad4102d41f1b4fd3
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/3


In [22]:
# best score

study.best_value

0.45875272767399033